# Patient Selection for Metabolic Modeling

This notebook identifies the most suitable patient from the dataset for initial modeling and reinforcement learning experimentation.

The goal is to select a patient with sufficient longitudinal data and rich metabolic dynamics to enable robust predictive modeling.

## Why start with a single patient?

Each individual with Type 1 Diabetes presents unique physiological dynamics:

- Different insulin sensitivity  
- Different meal timing and carbohydrate estimation  
- Different glucose variability  
- Different treatment modality  

Training a global model from all patients at once would blur individual dynamics and make debugging more difficult.

Therefore, we adopt a **single-patient modeling strategy** for the first phase of the project.

This allows:

- Clear visualization of glucose dynamics  
- More accurate predictive modeling  
- Construction of a stable metabolic simulator  
- Controlled reinforcement learning experimentation  

Multi-patient modeling will be considered in later stages.

## Selection Criteria

To identify the most suitable patient, we evaluate the following metrics for each dataset file:

1. **Number of recorded days**  
   Ensures sufficient longitudinal data for training and validation.

2. **Glucose standard deviation (glucose_std)**  
   Measures variability. Higher variability provides richer dynamic information.

3. **Glucose range (max - min)**  
   Indicates the presence of hyperglycemic and hypoglycemic excursions.

4. **Number of meal events (carb_input > 0)**  
   Captures postprandial glucose dynamics.

5. **Number of bolus insulin events**  
   Captures active glucose correction behavior.

The patient with strong variability, sufficient duration, and frequent interventions will be selected.

In [1]:
import pandas as pd
import os

data_path = "../data/"
files = [f for f in os.listdir(data_path) if f.endswith(".csv")]

summary = []

for file in files:
    df = pd.read_csv(os.path.join(data_path, file), sep=";")
    df["time"] = pd.to_datetime(df["time"])
    df = df.sort_values("time")
    
    days = (df["time"].max() - df["time"].min()).days
    rows = len(df)
    glucose_std = df["glucose"].std()
    glucose_range = df["glucose"].max() - df["glucose"].min()
    meals = (df["carb_input"] > 0).sum()
    bolus = (df["bolus_volume_delivered"] > 0).sum()
    
    summary.append([
        file,
        days,
        rows,
        round(glucose_std,2),
        round(glucose_range,2),
        meals,
        bolus
    ])

summary_df = pd.DataFrame(summary, columns=[
    "file",
    "days",
    "rows",
    "glucose_std",
    "glucose_range",
    "meals",
    "bolus"
])

summary_df.sort_values("glucose_std", ascending=False)

,file,days,rows,glucose_std,glucose_range,meals,bolus
5,HUPA0006P.csv,7,2290,84.74,396.0,37,46
3,HUPA0004P.csv,11,3184,83.02,371.0,20,45
12,HUPA0016P.csv,13,3835,79.84,322.0,83,91
16,HUPA0020P.csv,9,2862,78.79,350.0,0,7
6,HUPA0007P.csv,13,3857,78.66,345.0,82,89
10,HUPA0014P.csv,13,3829,72.46,403.5,40,41
0,HUPA0001P.csv,14,4096,70.64,404.0,40,76
14,HUPA0018P.csv,13,3895,70.27,318.0,0,0
13,HUPA0017P.csv,12,3599,69.45,361.0,35,25
22,HUPA0026P.csv,140,40605,68.67,382.0,46,334


## Selected Patient

After screening all patients, **HUPA0016P** was selected based on the following characteristics:

- 13 days of recorded data  
- Glucose standard deviation ≈ 79.84 mg/dL  
- Glucose range ≈ 322 mg/dL  
- 83 recorded meal events  
- 91 recorded bolus insulin events  

This patient provides:

- Strong metabolic variability  
- Frequent behavioral interventions  
- Rich dynamic structure  
- Sufficient longitudinal coverage  

These properties make HUPA0016P an ideal candidate for:

- Learning a personalized glucose prediction model  
- Constructing a data-driven metabolic simulator  
- Training a reinforcement learning agent  

The next step will be detailed exploratory analysis of this patient's data.